In [1]:
import tensorflow as tf

In [2]:
import pandas as pd
import numpy as np

print("=== SKELL'S GREENHOUSE: GENERADOR DE ENTRENAMIENTO ===")

# 1. Recolección de parámetros mediante inputs
temp_min = float(input("Temperatura Ambiente Mínima (°C) [Ej. 18]: "))
temp_max = float(input("Temperatura Ambiente Máxima (°C) [Ej. 30]: "))

hum_aire_min = float(input("Humedad Ambiente Mínima (%) [Ej. 60]: "))
hum_aire_max = float(input("Humedad Ambiente Máxima (%) [Ej. 80]: "))

hum_suelo_min = float(input("Humedad de la Tierra Mínima (%) [Ej. 25]: "))
hum_suelo_max = float(input("Humedad de la Tierra Máxima (%) [Ej. 75]: "))

luz_max = float(input("Luz Solar Analógica Máxima [Ej. 4095]: "))

nivel_tanque_min = float(input("Nivel Mínimo Operativo del Tanque (%) [Ej. 50]: "))

num_muestras = int(input("¿Cuántas filas de datos sintéticos deseas generar? [Ej. 5000]: "))

print("\nGenerando dataset sintético, por favor espera...")

# 2. Generación de las Entradas (Variables X) usando distribuciones aleatorias
np.random.seed(42) # Semilla para que los resultados sean reproducibles

data = {
    'Temperatura_C': np.random.uniform(temp_min, temp_max, num_muestras).round(1),
    'Humedad_Aire_PCT': np.random.uniform(hum_aire_min, hum_aire_max, num_muestras).round(1),
    'Humedad_Suelo_PCT': np.random.uniform(0, 100, num_muestras).round(1), # De 0 a 100 para que haya casos críticos
    'Luz_Analog': np.random.randint(0, luz_max + 1, num_muestras),
    'Nivel_Tanque_PCT': np.random.uniform(0, 100, num_muestras).round(1) # De 0 a 100 para entrenar la restricción
}

df = pd.DataFrame(data)

# 3. Lógica de las Salidas (Variables Y) - "El Cerebro"
# BOMBA DE AGUA (0 = Apagado, 1 = Encendido)
# Regla: Encender SI la humedad del suelo es menor al promedio esperado Y el tanque tiene agua suficiente.
humbral_riego = (hum_suelo_min + hum_suelo_max) / 2
df['Bomba_Agua'] = np.where((df['Humedad_Suelo_PCT'] <= humbral_riego) & (df['Nivel_Tanque_PCT'] >= nivel_tanque_min), 1, 0)

# LUZ UV (PWM 0 a 255)
# Regla: Proporcionalidad inversa a la luz solar. Si luz solar = 0, PWM = 255. Si luz solar = max, PWM = 0.
df['Luz_UV_PWM'] = 255 - (df['Luz_Analog'] * 255 / luz_max)
df['Luz_UV_PWM'] = df['Luz_UV_PWM'].clip(0, 255).astype(int) # Asegurar que no pase de 255 o baje de 0

# 4. Mostrar y Exportar
print("\n¡Dataset generado con éxito!")
print("\nPrimeras 5 filas de muestra:")
display(df.head(10)) # display() funciona perfectamente en Jupyter para mostrar tablas bonitas

archivo_csv = 'skells_greenhouse_dataset.csv'
df.to_csv(archivo_csv, index=False)
print(f"\nEl archivo ha sido guardado en tu carpeta actual como: {archivo_csv}")

=== SKELL'S GREENHOUSE: GENERADOR DE ENTRENAMIENTO ===


Temperatura Ambiente Mínima (°C) [Ej. 18]:  18
Temperatura Ambiente Máxima (°C) [Ej. 30]:  30
Humedad Ambiente Mínima (%) [Ej. 60]:  60
Humedad Ambiente Máxima (%) [Ej. 80]:  80
Humedad de la Tierra Mínima (%) [Ej. 25]:  25
Humedad de la Tierra Máxima (%) [Ej. 75]:  75
Luz Solar Analógica Máxima [Ej. 4095]:  4095
Nivel Mínimo Operativo del Tanque (%) [Ej. 50]:  50
¿Cuántas filas de datos sintéticos deseas generar? [Ej. 5000]:  5000



Generando dataset sintético, por favor espera...

¡Dataset generado con éxito!

Primeras 5 filas de muestra:


,Temperatura_C,Humedad_Aire_PCT,Humedad_Suelo_PCT,Luz_Analog,Nivel_Tanque_PCT,Bomba_Agua,Luz_UV_PWM
0,22.5,67.9,37.4,923,24.1,0,197
1,29.4,69.5,33.3,2325,27.4,0,110
2,26.8,77.1,17.6,3052,92.1,1,64
3,25.2,66.8,60.7,15,7.7,0,254
4,19.9,77.4,47.7,3596,61.0,1,31
5,19.9,61.8,86.6,2403,68.2,0,105
6,18.7,75.5,3.2,355,50.6,1,232
7,28.4,77.0,64.4,829,69.4,0,203
8,25.2,63.6,76.3,4010,17.3,0,5
9,26.5,68.6,75.9,2646,85.3,0,90



El archivo ha sido guardado en tu carpeta actual como: skells_greenhouse_dataset.csv


In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargar el dataset de Skell's GreenHouse
df = pd.read_csv('skells_greenhouse_dataset.csv')

# 2. Separar las Entradas (X) de las Salidas (Y) con los nombres CORREGIDOS
X = df[['Temperatura_C', 'Humedad_Aire_PCT', 'Humedad_Suelo_PCT', 'Luz_Analog', 'Nivel_Tanque_PCT']]
Y = df[['Bomba_Agua', 'Luz_UV_PWM']] 

# --- INTERACTIVIDAD CON EL USUARIO ---
print("🌱 --- Configuración de Entrenamiento de Skell's GreenHouse --- 🌱\n")

# Pedir porcentaje de validación (test size)
try:
    entrada_pct = input("¿Qué porcentaje de datos quieres ocultar para el examen final? (Ej. 20 para 20%): ")
    pct_oculto = float(entrada_pct)
    test_size_val = pct_oculto / 100.0
except ValueError:
    print("⚠️ Valor no válido. Usando 20% por defecto.")
    pct_oculto = 20.0
    test_size_val = 0.2

# Pedir cantidad de ciclos (epochs)
try:
    entrada_ciclos = input("¿Cuántos ciclos (epochs) de entrenamiento deseas ejecutar? (Ej. 50): ")
    ciclos = int(entrada_ciclos)
except ValueError:
    print("⚠️ Valor no válido. Usando 50 ciclos por defecto.")
    ciclos = 50
print("-" * 50)
# -------------------------------------

# 3. Dividir los datos usando el valor introducido
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size_val, random_state=42)

# 4. Normalizar los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Construir la Arquitectura de la Red Neuronal
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(5,)), 
    tf.keras.layers.Dense(8, activation='relu'),                    
    tf.keras.layers.Dense(2)                                        
])

# 6. Compilar el modelo
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 7. ¡Entrenar la IA!
print(f"\n🚀 Iniciando el entrenamiento con {ciclos} ciclos.")
print(f"📦 Reservando el {pct_oculto}% de los datos para la prueba final...\n")

history = model.fit(X_train_scaled, Y_train, epochs=ciclos, validation_split=0.2)

print("\n✅ ¡Entrenamiento de Skell's GreenHouse finalizado!")

🌱 --- Configuración de Entrenamiento de Skell's GreenHouse --- 🌱



¿Qué porcentaje de datos quieres ocultar para el examen final? (Ej. 20 para 20%):  10
¿Cuántos ciclos (epochs) de entrenamiento deseas ejecutar? (Ej. 50):  50


--------------------------------------------------

🚀 Iniciando el entrenamiento con 50 ciclos.
📦 Reservando el 10.0% de los datos para la prueba final...

Epoch 1/50


C:\Users\Windows 11\Desktop\GreenHouse\src\red\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 10630.0371 - mae: 63.1558 - val_loss: 10238.5293 - val_mae: 61.6205
Epoch 2/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 10024.3945 - mae: 61.1023 - val_loss: 9206.0840 - val_mae: 58.0375
Epoch 3/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8293.4805 - mae: 55.0842 - val_loss: 6793.2715 - val_mae: 49.3699
Epoch 4/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5237.5474 - mae: 43.1859 - val_loss: 3473.1680 - val_mae: 35.0987
Epoch 5/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2190.4822 - mae: 27.8177 - val_loss: 1112.4674 - val_mae: 20.1816
Epoch 6/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 632.7130 - mae: 15.1537 - val_loss: 321.8191 - val_mae: 10.9153
Epoch 7/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 244.7533 - mae: 9.0294 - val_loss: 180.3774 - val_mae: 7.6522
Epoch 8/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 174.3126 - mae: 7.2688 - val_loss: 149.1584 - val_mae: 6.7999
Epoch 9/50

In [7]:
# 8. Evaluar el modelo con los datos ocultos (El "Examen Final")
loss, mae = model.evaluate(X_test_scaled, Y_test, verbose=0)
print("📊 --- Resultados del Examen Final ---")
print(f"Margen de Error Promedio (MAE): {mae:.4f}")

# 9. Guardar el modelo en formato estándar de Keras (Para PC)
modelo_keras = "skells_greenhouse_model.keras"
model.save(modelo_keras)
print(f"\n💾 ¡Modelo estándar guardado como '{modelo_keras}'!")

# 10. CONVERTIR A TENSORFLOW LITE (Crucial para el ESP32)
print("\n⚙️ Convirtiendo el modelo para microcontroladores (TinyML)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 11. Guardar el archivo .tflite
modelo_tflite = "skells_greenhouse.tflite"
with open(modelo_tflite, 'wb') as f:
    f.write(tflite_model)
print(f"📱 ¡ÉXITO! Modelo optimizado guardado como '{modelo_tflite}'")

📊 --- Resultados del Examen Final ---
Margen de Error Promedio (MAE): 0.6962

💾 ¡Modelo estándar guardado como 'skells_greenhouse_model.keras'!

⚙️ Convirtiendo el modelo para microcontroladores (TinyML)...
INFO:tensorflow:Assets written to: C:\Users\WINDOW~1\AppData\Local\Temp\tmp10cifxmm\assets


INFO:tensorflow:Assets written to: C:\Users\WINDOW~1\AppData\Local\Temp\tmp10cifxmm\assets


Saved artifact at 'C:\Users\WINDOW~1\AppData\Local\Temp\tmp10cifxmm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 5), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2401003923088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2401003914832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2401003925200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2401003925008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2401003926352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2401003915984: TensorSpec(shape=(), dtype=tf.resource, name=None)
📱 ¡ÉXITO! Modelo optimizado guardado como 'skells_greenhouse.tflite'


In [8]:
# 12. Convertir el binario .tflite a un archivo de cabecera C++ (.h)
def hex_to_c_array(tflite_file, variable_name="skells_model"):
    with open(tflite_file, 'rb') as f:
        data = f.read()
    
    # Escribir el archivo .h
    with open(f"{variable_name}.h", "w") as f_out:
        f_out.write(f'const unsigned char {variable_name}[] __attribute__((aligned(16))) = {{\n  ')
        
        for i, byte in enumerate(data):
            f_out.write(f'0x{byte:02x}')
            if i < len(data) - 1:
                f_out.write(', ')
            if (i + 1) % 12 == 0:
                f_out.write('\n  ')
        
        f_out.write('\n};\n\n')
        f_out.write(f'const int {variable_name}_len = {len(data)};\n')
    
    print(f"✅ ¡Archivo '{variable_name}.h' generado exitosamente!")
    print(f"📦 Tamaño final en bytes: {len(data)}")

# Ejecutar la conversión
hex_to_c_array("skells_greenhouse.tflite")

✅ ¡Archivo 'skells_model.h' generado exitosamente!
📦 Tamaño final en bytes: 3012
